In [ ]:
import json

from urllib.request import urlopen
from PIL import Image
import torch
import numpy as np
from huggingface_hub import hf_hub_download
from open_clip import create_model_and_transforms, get_tokenizer
from open_clip.factory import HF_HUB_PREFIX, _MODEL_CONFIGS

LOCAL_DIR = "/data/ckpt/BiomedCLIP"
model_name = "biomedclip_local"
with open(F"{LOCAL_DIR}/open_clip_config.json", "r") as f:
    config = json.load(f)
    model_cfg = config["model_cfg"]
    preprocess_cfg = config["preprocess_cfg"]

# 注册模型配置
if (not model_name.startswith(HF_HUB_PREFIX)
    and model_name not in _MODEL_CONFIGS
    and config is not None):
    _MODEL_CONFIGS[model_name] = model_cfg

# 创建模型和预处理函数
model, _, preprocess = create_model_and_transforms(
    model_name=model_name,
    pretrained=f"{LOCAL_DIR}/open_clip_pytorch_model.bin",
    **{f"image_{k}": v for k, v in preprocess_cfg.items()},
)

# 设置设备并将模型移到设备上
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

# 创建随机图像
random_array = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
image = Image.fromarray(random_array)

# 图像预处理
processed_image = preprocess(image).unsqueeze(0).to(device)

# 提取特征
with torch.no_grad():
    image_features = model.encode_image(processed_image)
    normalized_features = image_features / image_features.norm(dim=1, keepdim=True)

# 输出结果
features_np = normalized_features.cpu().numpy()
print(f"特征维度: {features_np.shape}")
print(f"特征前5值: {features_np[0, :5]}")
print(f"特征范数: {np.linalg.norm(features_np):.4f}")

OSError: We couldn't connect to 'https://huggingface.co' to load this file, couldn't find it in the cached files and it looks like microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract is not the path to a directory containing a file named config.json.
Checkout your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.